# Patient Harm Reported with Patient-Transfer and Lifting Devices: FDA MAUDE

The OSHA modules in this repository describe injuries to staff. This module describes the other side: injuries to patients, reported to FDA, in events involving devices used to lift, transfer, or transport patients.

**Author** Yuxuan Huang · Vantara Medical Equipment (Mahoraga Vantara LLC)
**Data** FDA Manufacturer and User Facility Device Experience (MAUDE), via the openFDA API
**Source data** US Government work, 17 U.S.C. §105

### Disclosure

The author distributes a powered patient transfer system. No product, brand, or manufacturer is named in any published output of this module. Devices are grouped by FDA product code and device type only. Brand and manufacturer fields are dropped before anything is written to `derived/`.

### What this data can and cannot show

- **Not rates.** MAUDE is a passive reporting system. Many events are never reported, and FDA does not know how often each device is used. Counts cannot be read as risk.
- **Not cause.** FDA states that a report does not establish that a device caused or contributed to an event.
- **Only device-involved events.** A patient harmed during a transfer done by hand, with no device, does not appear here.
- **Reporting changes over time.** Counts by year reflect changes in reporting rules and practice as well as changes in events. 2015 shows an unusually high number of malfunction reports and reports with no event type; it is not used for trend comparison.

### How the questions are answered

The questions and definitions were fixed in [`analysis-plan.md`](analysis-plan.md) before the second review sample was drawn. Each answer rests on a blind human review of 400 reports drawn at random (Samples A and B), not on keyword rules. Keyword rules for patient falls are tested against the review and their accuracy reported.

## 1. Environment

In [1]:
!pip install -q pandas requests
import os, re, time, json, requests, pandas as pd, numpy as np
pd.set_option('display.max_columns', None); pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)
os.makedirs('data', exist_ok=True); os.makedirs('derived', exist_ok=True)
API = 'https://api.fda.gov/device'
# openFDA allows 1,000 requests a day per IP without a key, and Colab IPs are shared, so a key is needed.
# Get a free key at https://open.fda.gov/apis/authentication/ and paste it below (do not publish the key).
API_KEY = ''   # <- paste your key here before running; clear it before saving the notebook to GitHub
def get(endpoint, params, tries=6):
    p = dict(params)
    if API_KEY: p['api_key'] = API_KEY
    for i in range(tries):
        r = requests.get(f'{API}/{endpoint}.json', params=p, timeout=120)
        if r.status_code == 404: return {'results': [], 'meta': {'results': {'total': 0}}}
        if r.status_code in (403, 429): time.sleep(10 * (i + 1)); continue
        r.raise_for_status(); return r.json()
    raise RuntimeError(f'openFDA returned {r.status_code}. Without an API key, shared Colab addresses run out of quota; add API_KEY above.')

## 2. Find the device product codes

FDA assigns every device type a product code. Rather than hard-coding codes, the notebook searches FDA's device classification list for device names that describe lifting, transferring, or transporting patients, and prints every match. Section 2b lists codes to exclude after reading the printed list; the final list used is written to `derived/product-codes.csv`.

In [2]:
NAME_RX = r"\b(?:lift|hoist|sling|transfer|stretcher|litter|gurney|transport|slide|sliding|positioner|patient mover|air[- ]?assist|turning)\b"
seen = {}
for term in ['lift', 'hoist', 'sling', 'transfer', 'stretcher', 'litter', 'transport', 'slide', 'positioner', 'turning', 'mover']:
    skip = 0
    while True:
        js = get('classification', {'search': f'device_name:"{term}"', 'limit': 1000, 'skip': skip})
        res = js.get('results', [])
        for x in res:
            seen[x['product_code']] = x
        if len(res) < 1000: break
        skip += 1000
cls = pd.DataFrame(seen.values())[['product_code', 'device_name', 'device_class', 'regulation_number', 'medical_specialty_description']]
cls = cls[cls.device_name.str.lower().str.contains(NAME_RX, regex=True)].sort_values('device_name').reset_index(drop=True)
print(len(cls), 'candidate product codes'); cls

55 candidate product codes


,product_code,device_name,device_class,regulation_number,medical_specialty_description
0,IKX,"Aid, Transfer",1,890.5050,Physical Medicine
1,GNC,"Antigens, Febrile, Slide And Tube, All Groups, Salmonella Spp.",2,866.3550,Microbiology
2,GSI,"Antigens, Slide And Tube, All Types, Listeria Monocytogenes",1,866.3355,Microbiology
3,GSL,"Antigens, Slide And Tube, Francisella Tularensis",2,866.3280,Microbiology
4,RDV,"Automotive Accessory, Automobile Or Transport Vehicle",N,,Unknown
5,LGN,"Chamber, Patient Transport Isolation",2,880.5450,General Hospital
6,KIY,"Chamber, Slide Culture",1,864.2240,Hematology
7,ONB,Closed Antineoplastic And Hazardous Drug Reconstitution And Transfer System,2,880.5440,General Hospital
8,KDK,"Container, Transport, Kidney",2,876.5880,"Gastroenterology, Urology"
9,KES,"Coverslips, Microscope Slide",1,864.3010,Pathology


### 2b. Codes used

The search returns many devices that share a word but have nothing to do with moving patients (microscope slides, specimen transport media, arm slings). The codes below are the ones whose FDA device name describes lifting, transferring, or transporting a patient's body. Every other candidate is written to `derived/product-codes.csv` as excluded, with the reason.

In [3]:
INCLUDE = {
    'FRZ': 'Device, Patient Transfer, Powered (21 CFR 880.6775)',
    'FMR': 'Device, Transfer, Patient, Manual (21 CFR 880.6785)',
    'IKX': 'Aid, Transfer (21 CFR 890.5050)',
    'FNG': 'Lift, Patient, AC-Powered (21 CFR 880.5500)',
    'FSA': 'Lift, Patient, Non-AC-Powered (21 CFR 880.5510)',
    'FPO': 'Stretcher, Wheeled (21 CFR 880.6910)',
    'INK': 'Stretcher, Wheeled, Powered (21 CFR 890.3690)',
    'FPP': 'Stretcher, Hand-Carried (21 CFR 880.6900)',
    'ILK': 'Transport, Patient, Powered (21 CFR 890.5150)',
}
GROUP = {'FRZ': 'transfer device', 'FMR': 'transfer device', 'IKX': 'transfer device',
         'FNG': 'patient lift', 'FSA': 'patient lift',
         'FPO': 'stretcher', 'INK': 'stretcher', 'FPP': 'stretcher', 'ILK': 'powered transport'}
cls['used'] = cls.product_code.isin(INCLUDE)
cls['excluded_reason'] = np.where(cls.used, '', 'device does not lift, transfer, or transport a patient')
cls['group'] = cls.product_code.map(GROUP).fillna('')
cls.to_csv('derived/product-codes.csv', index=False)
codes = cls[cls.used].copy()
missing = set(INCLUDE) - set(codes.product_code)
print(len(codes), 'codes used', '| not found in search:', missing or 'none')
codes[['product_code', 'device_name', 'regulation_number', 'group']]

9 codes used | not found in search: none


,product_code,device_name,regulation_number,group
0,IKX,"Aid, Transfer",890.5050,transfer device
15,FRZ,"Device, Patient Transfer, Powered",880.6775,transfer device
16,FMR,"Device, Transfer, Patient, Manual",880.6785,transfer device
24,FNG,"Lift, Patient, Ac-Powered",880.5500,patient lift
25,FSA,"Lift, Patient, Non-Ac-Powered",880.5510,patient lift
42,FPP,"Stretcher, Hand-Carried",880.6900,stretcher
44,FPO,"Stretcher, Wheeled",880.6910,stretcher
45,INK,"Stretcher, Wheeled, Powered",890.3690,stretcher
52,ILK,"Transport, Patient, Powered",890.5150,powered transport


## 3. Download event reports

All reports received from 2015 through 2025 for the codes above, one request window per code and year (openFDA returns at most 26,000 records per query). Brand, model, and manufacturer fields are not kept.

In [4]:
def events(code, y0, y1):
    out, skip = [], 0
    q = f'device.device_report_product_code:"{code}" AND date_received:[{y0}0101 TO {y1}1231]'
    while True:
        js = get('event', {'search': q, 'limit': 1000, 'skip': skip})
        res = js.get('results', [])
        for e in res:
            dev = (e.get('device') or [{}])[0]
            out.append({
                'report_id': e.get('mdr_report_key'), 'date_received': e.get('date_received'),
                'event_type': e.get('event_type'), 'product_code': code,
                'generic_name': dev.get('generic_name'),
                'patient_outcome': ';'.join(sorted({o for p in (e.get('patient') or []) for o in (p.get('sequence_number_outcome') or []) if o})),
                'text': ' '.join(t.get('text', '') for t in (e.get('mdr_text') or []) if t.get('text')),
            })
        if len(res) < 1000 or skip + 1000 >= 26000: break
        skip += 1000
    return out
def total(code, y0, y1):
    q = f'device.device_report_product_code:"{code}" AND date_received:[{y0}0101 TO {y1}1231]'
    return get('event', {'search': q, 'limit': 1})['meta']['results']['total']
rows = []
for c in codes.product_code:
    n_all = total(c, 2015, 2025)
    print(f'{c}: {n_all:,} reports, 2015-2025')
    if n_all == 0: continue
    years = [(2015, 2025)] if n_all <= 25000 else [(y, y) for y in range(2015, 2026)]
    for y0, y1 in years:
        rows += events(c, y0, y1)
ev = pd.DataFrame(rows).drop_duplicates('report_id')
ev['year'] = ev.date_received.str[:4].astype(int)
ev['group'] = ev.product_code.map(GROUP)
ev['text_l'] = ev.text.fillna('').str.lower()
print(f'{len(ev):,} reports'); ev.event_type.value_counts()

IKX: 192 reports, 2015-2025
FRZ: 23 reports, 2015-2025
FMR: 81 reports, 2015-2025
FNG: 93 reports, 2015-2025
FSA: 5,247 reports, 2015-2025
FPP: 650 reports, 2015-2025
FPO: 7,214 reports, 2015-2025
INK: 882 reports, 2015-2025
ILK: 57 reports, 2015-2025
14,439 reports


,count
event_type,
Malfunction,11963
Injury,2115
Death,261
,81
Other,19


## 4. Reports by year and event type

In [5]:
by_year = ev.groupby(['year', 'event_type']).size().unstack(1).fillna(0).astype(int)
by_year

event_type,,Death,Injury,Malfunction,Other
year,,,,,
2015,42,35,201,1722,3
2016,34,21,205,724,2
2017,2,22,235,473,0
2018,1,25,190,972,0
2019,0,24,197,983,0
2020,1,26,240,846,0
2021,0,20,220,1079,0
2022,0,17,141,990,0
2023,1,22,159,1413,10


## 5. The two review samples

Sample A (200 reports, `random_state=2026`) and Sample B (200 further reports, `random_state=2027`, drawn after the analysis plan was committed). Report text names manufacturers and products, so the review copies with text are written to `data/` and are not published. The labels are published in `derived/review-sample-A.csv` and `derived/review-sample-B.csv`, keyed by report number; anyone can read a report's text on FDA's MAUDE site by its number.

In [6]:
URL = 'https://raw.githubusercontent.com/yuxuan-huang-data/us-care-safety-signals/main/maude-transfer-devices/derived/'
def labels(name):
    p = f'derived/{name}'
    return pd.read_csv(p if os.path.exists(p) else URL + name, dtype=str).fillna('')
la, lb = labels('review-sample-A.csv'), labels('review-sample-B.csv')
ev['report_id'] = ev.report_id.astype(str)
# The samples were drawn with ev.sample(200, random_state=2026) and, from the remainder, random_state=2027.
# They are selected here by their published report numbers, so the analysis does not depend on FDA's file staying identical.
cols = ['report_id', 'event_type', 'text']
A = la.merge(ev[cols], on='report_id', how='left'); B = lb.merge(ev[cols], on='report_id', how='left')
A[cols].to_csv('data/review-sample-A-with-text.csv', index=False)   # internal: names manufacturers
B[cols].to_csv('data/review-sample-B-with-text.csv', index=False)
print('Sample A reports found:', A.text.notna().sum(), 'of', len(A), '| Sample B:', B.text.notna().sum(), 'of', len(B))

Sample A reports found: 200 of 200 | Sample B: 200 of 200


## 6. Question 1: scale

Share of reports in which a patient was injured, with a Wilson 95% interval, applied to all reports.

In [7]:
def wilson(k, n, z=1.96):
    p = k / n; c = (p + z*z/(2*n)) / (1 + z*z/n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return p, c - h, c + h
N = len(ev); rows = []
for name, d in [('A', A), ('B', B), ('A+B', pd.concat([A, B], ignore_index=True))]:
    k = int((d.who == 'patient').sum()); p, lo, hi = wilson(k, len(d))
    rows.append({'sample': name, 'reports': len(d), 'patient injured': k, 'share': round(p, 3),
                 'ci95_low': round(lo, 3), 'ci95_high': round(hi, 3),
                 'implied reports': round(N * p), 'implied low': round(N * lo), 'implied high': round(N * hi),
                 'staff injured': int((d.who == 'staff').sum())})
scale = pd.DataFrame(rows); scale.to_csv('derived/q1-scale.csv', index=False); scale

,sample,reports,patient injured,share,ci95_low,ci95_high,implied reports,implied low,implied high,staff injured
0,A,200,32,0.16,0.116,0.217,2310,1670,3135,8
1,B,200,40,0.20,0.150,0.261,2888,2172,3767,15
2,A+B,400,72,0.18,0.145,0.221,2599,2100,3186,23


## 7. Question 2: stage

Among reports in which anyone was injured. Counts only; the number of injured reports is too small to estimate national shares.

In [8]:
INJ = ['patient', 'staff', 'visitor', 'unclear']
inj = pd.concat([A[A.who.isin(INJ)].assign(sample='A'), B[B.who.isin(INJ)].assign(sample='B')], ignore_index=True)
stage = pd.crosstab(inj.stage, inj['sample'], margins=True, margins_name='total')
pi = inj[inj.who == 'patient'].stage.value_counts(); stage['patient injured only'] = pi; stage.loc['total', 'patient injured only'] = pi.sum()
stage = stage.fillna(0).astype(int); stage.to_csv('derived/q2-stage.csv'); stage

sample,A,B,total,patient injured only
stage,,,,
accessory,2,4,6,5
device_defect,14,23,37,27
environment,1,1,2,2
maintenance_age,5,2,7,3
patient_behavior,1,2,3,2
unclear,6,2,8,7
use_training,16,21,37,26
total,45,55,100,72


## 8. Question 3: lateral transfers

In [9]:
both = pd.concat([A.assign(sample='A'), B.assign(sample='B')], ignore_index=True)
task = pd.crosstab(both.task, both['sample'], margins=True, margins_name='total')
k = int((both.task == 'lateral').sum()); p, lo, hi = wilson(k, len(both))
print(f'lateral transfers: {k} of {len(both)} ({p:.2%}; 95% CI {lo:.2%}-{hi:.2%})')
task.to_csv('derived/q3-task.csv'); task

lateral transfers: 3 of 400 (0.75%; 95% CI 0.26%-2.18%)


sample,A,B,total
task,,,
lateral,1,2,3
lift_sling,40,78,118
not_stated,85,0,85
not_transfer,40,25,65
repositioning,2,0,2
sit_to_stand,4,4,8
stretcher_transport,28,91,119
total,200,200,400


## 9. Question 4: patient-fall rules, tested on Sample B

The original rule was written before any review. The revised rule was written while reading Sample A, as specified in the analysis plan; its Sample A score is a development score only. Both are scored on Sample B against the review question "did the patient fall, slide from, or drop out of the device", answered whether or not the patient was injured. Reports marked `unclear` are excluded.

In [10]:
ORIG = r"\b(?:fell|fall(?:s|en|ing)?|dropped|slid (?:off|out|down)|slipped (?:off|out)|tipped over|lowered to the floor)\b"
PAT   = r"(?:patient|pt|resident|client|individual|user|occupant|he|she|they)"
FALLV = r"(?:fell|fall|fallen|dropped|slid (?:off|out|down|from)|slipped (?:off|out|from|through)|came out of the sling|landed on the floor|was lowered to the floor)"
STAFF = r"(?:staff|nurse|caregiver|aide|technician|tech|visitor|family member|crew|paramedic|emt|operator|attendant)s?"
def revised(s):
    pos = (re.search(rf"\b{PAT}\b(?:\W+\w+){{0,6}}?\W+{FALLV}\b", s) or re.search(r"\bdropped (?:the )?(?:patient|resident)\b", s)
           or re.search(r"\b(?:patient|resident)'?s fall\b", s))
    if not pos: return False
    if (re.search(r"\b(?:no (?:patient )?falls?|did not fall|didn't fall|without (?:a )?fall|prevent\w* (?:\w+ ){0,3}(?:from )?fall\w*|avoid\w* (?:a )?fall)\b", s)
            and not re.search(r"\b(?:patient|resident|pt)\b(?:\W+\w+){0,3}?\W+(?:fell|was dropped|slid off)", s)):
        return False
    staff_fall = re.search(rf"\b{STAFF}\b(?:\W+\w+){{0,4}}?\W+(?:fell|fall|tripped|slipped)", s)
    pat_fall = re.search(rf"\b(?:patient|pt|resident|client|individual|occupant)\b(?:\W+\w+){{0,6}}?\W+{FALLV}\b", s)
    if staff_fall and not pat_fall: return False
    return True
d = B[B.fell != 'unclear'].copy(); t = d.text.fillna('').str.lower(); truth = d.fell == 'yes'
rows = []
for name, pred in [('original', t.str.contains(ORIG, regex=True)), ('revised', t.map(revised))]:
    tp = int((pred & truth).sum())
    rows.append({'rule': name, 'reports': len(d), 'patient fell (review)': int(truth.sum()), 'flagged': int(pred.sum()),
                 'correct': tp, 'precision': round(tp / pred.sum(), 3), 'recall': round(tp / truth.sum(), 3)})
rules = pd.DataFrame(rows); rules.to_csv('derived/q4-fall-rules.csv', index=False); rules

,rule,reports,patient fell (review),flagged,correct,precision,recall
0,original,198,42,49,35,0.714,0.833
1,revised,198,42,29,26,0.897,0.619


## 10. Question 5: how the reporter's own event type compares with the review

In [11]:
et = pd.crosstab(both.event_type.fillna('blank').replace('', 'blank'), both.who, margins=True, margins_name='total')
et.to_csv('derived/q5-event-type-vs-review.csv'); et

who,none,patient,staff,unclear,total
event_type,,,,,
Death,0,4,1,0,5
Injury,4,41,10,1,56
Malfunction,293,27,11,4,335
blank,3,0,1,0,4
total,300,72,23,5,400


## 11. Export

In [12]:
by_year.to_csv('derived/reports-by-year-type.csv')
print(sorted(os.listdir('derived')))

['product-codes.csv', 'q1-scale.csv', 'q2-stage.csv', 'q3-task.csv', 'q4-fall-rules.csv', 'q5-event-type-vs-review.csv', 'reports-by-year-type.csv']
